# SSO Signup Optimization — Data Analysis

This notebook builds up, CTE by CTE, the two queries used to establish the warehouse baselines for this experiment's power analyses: the **signup-page conversion baseline** (Visit-to-Signup Rate) and the **First Fix conversion baseline**. Each step below adds exactly one CTE and re-runs, so the final step in each part reproduces the exact query and numbers used for sizing.

Both queries share the same starting point: a visitor's first visit to the signup page each month. Part A builds the signup baseline on top of that; Part B reuses the same starting CTE and swaps in a different conversion signal to get the First Fix baseline.

In [1]:
import pandas as pd
from amphibian import get_data_accessor

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

def query(sql):
    return get_data_accessor(engines=['presto']).fetch_sql(sql=sql)

## Part A — Signup-page conversion baseline (Visit-to-Signup Rate)

Built up using the actual parameters used for sizing: visits from `2026-01-01` onward, to any URL starting with `https://www.stitchfix.com/signup`.

### A1 — the `signup_page_visits` CTE alone

One row per visitor per month, anchored to the **first** time they reached the signup page that month (not every visit — just the earliest one, which is what "new" is measured relative to).

In [2]:
query("""--sql
SELECT visitor_id, DATE_TRUNC('month', datetime_in_utc) AS month, MIN(datetime_in_utc) AS signup_page_ts
FROM curated.product_tracking_events
WHERE date_in_utc >= DATE '2026-01-01'
  AND url LIKE 'https://www.stitchfix.com/signup%'
GROUP BY visitor_id, DATE_TRUNC('month', datetime_in_utc)
ORDER BY signup_page_ts
LIMIT 5
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,visitor_id,month,signup_page_ts
0,4a4eda80-25ef-4e13-bc8c-1f9a1ccc66e7,2026-01-01 00:00:00.000,2026-01-01 00:00:03.030
1,15a3e35b-3505-4448-bd24-b11f0f715372,2026-01-01 00:00:00.000,2026-01-01 00:00:03.750
2,c7fff5a0-6feb-491e-824d-09bdd702c86e,2026-01-01 00:00:00.000,2026-01-01 00:00:22.407
3,0f1f2a01-4bde-40bf-a211-e3b4a040d8ff,2026-01-01 00:00:00.000,2026-01-01 00:00:25.349
4,9b4b1515-af03-4221-9410-b2f7dcf762c6,2026-01-01 00:00:00.000,2026-01-01 00:00:32.175


In [3]:
query("""--sql
SELECT DATE_TRUNC('month', datetime_in_utc) AS month, COUNT(DISTINCT visitor_id) AS distinct_visitors
FROM curated.product_tracking_events
WHERE date_in_utc >= DATE '2026-01-01'
  AND url LIKE 'https://www.stitchfix.com/signup%'
GROUP BY 1
ORDER BY 1 DESC
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,month,distinct_visitors
0,2026-07-01 00:00:00.000,299929
1,2026-06-01 00:00:00.000,259883
2,2026-05-01 00:00:00.000,311651
3,2026-04-01 00:00:00.000,334200
4,2026-03-01 00:00:00.000,423533
5,2026-02-01 00:00:00.000,386630
6,2026-01-01 00:00:00.000,412194


These monthly counts are **every** visitor who reached the signup page that month — before any eligibility gate is applied. June 2026's 259,883 is bigger than what the final query reports for June (225,286) because this hasn't yet excluded visitors who had already converted before this particular visit — that gate is added next.

### A2 — the `visitor_signup` CTE alone

One signup timestamp per visitor (`NULL` if they never signed up), from `curated.user_session_conversion_metrics`. This is a visitor-level fact independent of any specific page visit — it's what lets the next step tell whether a given signup-page visit was this visitor's *first* time converting, or a repeat visit from someone who already had an account.

In [4]:
query("""--sql
SELECT visitor_id, MAX(signup_ts) AS signup_ts
FROM curated.user_session_conversion_metrics
WHERE region = 'US'
  AND date_in_utc >= DATE '2026-01-01'
GROUP BY visitor_id
HAVING MAX(signup_ts) IS NOT NULL
ORDER BY signup_ts
LIMIT 5
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,visitor_id,signup_ts
0,9db97391-d09b-44af-a3fb-1ab9ced6dda1,2011-10-30 07:00:00.000
1,bef04acd-e6bb-4d5e-81e8-1c8f6194ae5c,2011-10-30 07:00:00.000
2,f8ea87b2-f2e0-4a41-b619-4ad40c0cb69b,2011-10-30 07:00:00.000
3,fd18e416-c9c1-4eb7-ab03-ae9104ffe0c8,2011-10-30 07:00:00.000
4,94ef380f-ac9b-4279-91df-d0d360a71451,2011-10-30 07:00:00.000


In [5]:
query("""--sql
SELECT COUNT(*) AS n_visitors, COUNT(signup_ts) AS n_with_signup
FROM (
  SELECT visitor_id, MAX(signup_ts) AS signup_ts
  FROM curated.user_session_conversion_metrics
  WHERE region = 'US'
    AND date_in_utc >= DATE '2026-01-01'
  GROUP BY visitor_id
)
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,n_visitors,n_with_signup
0,62064419,6331451


Only ~6.3M of the ~62M distinct visitors in this window ever signed up at all (most `visitor_id`s are one-off browsing sessions that never convert) — note `signup_ts` can be far in the past (2011 in the sample above), since it's the visitor's all-time signup date, not something scoped to this window.

### A3 — the `classified` CTE: joining the two together

Left join `signup_page_visits` to `visitor_signup` and derive two flags per visit:
- **`new_visitor`**: 1 if the visitor hadn't signed up yet as of this visit (i.e. this is a genuine "new" reach, not a returning member re-visiting the page).
- **`signup`**: 1 if they signed up within 7 days of this visit.

Five real visitors from A1's sample, run through this logic, cover all three cases:

In [6]:
query(f"""--sql
WITH signup_page_visits AS (
    SELECT visitor_id, DATE_TRUNC('month', datetime_in_utc) AS month, MIN(datetime_in_utc) AS signup_page_ts
    FROM curated.product_tracking_events
    WHERE date_in_utc >= DATE '2026-01-01'
      AND url LIKE 'https://www.stitchfix.com/signup%'
      AND visitor_id IN ('4a4eda80-25ef-4e13-bc8c-1f9a1ccc66e7', '15a3e35b-3505-4448-bd24-b11f0f715372', 'c7fff5a0-6feb-491e-824d-09bdd702c86e', '0f1f2a01-4bde-40bf-a211-e3b4a040d8ff', '9b4b1515-af03-4221-9410-b2f7dcf762c6')
    GROUP BY visitor_id, DATE_TRUNC('month', datetime_in_utc)
),
visitor_signup AS (
    SELECT visitor_id, MAX(signup_ts) AS signup_ts
    FROM curated.user_session_conversion_metrics
    WHERE region = 'US' AND date_in_utc >= DATE '2026-01-01'
      AND visitor_id IN ('4a4eda80-25ef-4e13-bc8c-1f9a1ccc66e7', '15a3e35b-3505-4448-bd24-b11f0f715372', 'c7fff5a0-6feb-491e-824d-09bdd702c86e', '0f1f2a01-4bde-40bf-a211-e3b4a040d8ff', '9b4b1515-af03-4221-9410-b2f7dcf762c6')
    GROUP BY visitor_id
)
SELECT
    v.visitor_id, v.signup_page_ts, s.signup_ts,
    CASE WHEN s.signup_ts IS NULL OR s.signup_ts >= v.signup_page_ts THEN 1 ELSE 0 END AS new_visitor,
    CASE WHEN s.signup_ts >= v.signup_page_ts
          AND s.signup_ts < v.signup_page_ts + INTERVAL '7' DAY
         THEN 1 ELSE 0 END AS signup
FROM signup_page_visits v
LEFT JOIN visitor_signup s ON s.visitor_id = v.visitor_id
ORDER BY v.signup_page_ts
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,visitor_id,signup_page_ts,signup_ts,new_visitor,signup
0,4a4eda80-25ef-4e13-bc8c-1f9a1ccc66e7,2026-01-01 00:00:03.030,2022-05-13 22:52:03.740,0,0
1,15a3e35b-3505-4448-bd24-b11f0f715372,2026-01-01 00:00:03.750,None,1,0
2,c7fff5a0-6feb-491e-824d-09bdd702c86e,2026-01-01 00:00:22.407,None,1,0
3,0f1f2a01-4bde-40bf-a211-e3b4a040d8ff,2026-01-01 00:00:25.349,2026-01-01 00:02:00.184,1,1
4,9b4b1515-af03-4221-9410-b2f7dcf762c6,2026-01-01 00:00:32.175,2026-01-01 00:00:46.555,1,1


Reading each row: visitor 1 has a `signup_ts` from **2022** — years before this visit, so `new_visitor = 0` (a returning member, not a fresh reach; excluded from the funnel entirely). Visitors 2 and 3 have `signup_ts = NULL` — never signed up, so `new_visitor = 1, signup = 0`. Visitors 4 and 5 signed up **minutes** after reaching the page — `new_visitor = 1, signup = 1`, a real conversion. This is exactly the population/eligibility logic the final query aggregates over.

### A4 — the full signup-page conversion query

Add `month_days` (distinct calendar days observed per month, for the daily-rate denominator) and the final aggregation. This is the complete query used for the signup-page conversion baseline.

In [7]:
traffic_query = """--sql
WITH signup_page_visits AS (
    SELECT
        visitor_id,
        DATE_TRUNC('month', datetime_in_utc) AS month,
        MIN(datetime_in_utc) AS signup_page_ts
    FROM curated.product_tracking_events
    WHERE date_in_utc >= DATE '2026-01-01'
      AND url LIKE 'https://www.stitchfix.com/signup%'
    GROUP BY visitor_id, DATE_TRUNC('month', datetime_in_utc)
),
visitor_signup AS (
    SELECT
        visitor_id,
        MAX(signup_ts) AS signup_ts
    FROM curated.user_session_conversion_metrics
    WHERE region = 'US'
      AND date_in_utc >= DATE '2026-01-01'
    GROUP BY visitor_id
),
classified AS (
    SELECT
        v.month,
        v.signup_page_ts,
        CASE WHEN s.signup_ts IS NULL OR s.signup_ts >= v.signup_page_ts
             THEN 1 ELSE 0 END AS new_visitor,
        CASE WHEN s.signup_ts >= v.signup_page_ts
              AND s.signup_ts <  v.signup_page_ts + INTERVAL '7' DAY
             THEN 1 ELSE 0 END AS signup
    FROM signup_page_visits v
    LEFT JOIN visitor_signup s ON s.visitor_id = v.visitor_id
),
month_days AS (
    SELECT month, COUNT(DISTINCT CAST(signup_page_ts AS DATE)) AS days_observed
    FROM signup_page_visits
    GROUP BY month
)
SELECT
    c.month,
    md.days_observed,
    SUM(new_visitor) AS signup_page_visitors,
    SUM(signup) AS signups,
    CAST(SUM(signup) AS DOUBLE) / NULLIF(SUM(new_visitor), 0) AS signup_page_conv_rate,
    ROUND(SUM(new_visitor) / md.days_observed, 1) AS signup_page_visitors_per_day
FROM classified c
JOIN month_days md ON c.month = md.month
GROUP BY c.month, md.days_observed
ORDER BY c.month DESC
"""

query(traffic_query)

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,month,days_observed,signup_page_visitors,signups,signup_page_conv_rate,signup_page_visitors_per_day
0,2026-07-01 00:00:00.000,28,256861,132959,0.517630,9173
1,2026-06-01 00:00:00.000,30,225286,116552,0.517351,7509
2,2026-05-01 00:00:00.000,31,269155,141919,0.527276,8682
3,2026-04-01 00:00:00.000,30,288796,153013,0.529831,9626
4,2026-03-01 00:00:00.000,31,368711,199606,0.541362,11893
5,2026-02-01 00:00:00.000,28,335704,184453,0.549451,11989
6,2026-01-01 00:00:00.000,31,358841,199058,0.554725,11575


Signup-page conversion sits around **52–55%**, drifting down through the year — consistent with the earlier funnel-level counts once the "already converted" visitors from A1 are excluded.

## Part B — First Fix conversion baseline

Reuses the same `signup_page_visits` CTE from Part A unchanged — same visitors, same monthly anchor. What's different is the conversion signal: instead of asking "did they sign up," this asks "did they request a First Fix within 7 days," using `curated.user_session_conversion_metrics.request_7d_flag`.

### B1 — the `visitor_conversion` CTE alone

Same table as A2, but now also pulling `request_7d_flag` — the table's own precomputed "requested a First Fix within 7 days" flag, aggregated per visitor via `MAX()` (a visitor can have multiple sessions in this table; this is 1 if *any* of them carries the flag).

In [8]:
query("""--sql
SELECT visitor_id, MAX(signup_ts) AS signup_ts, MAX(COALESCE(request_7d_flag, 0)) AS first_fix_request_7d_flag
FROM curated.user_session_conversion_metrics
WHERE region = 'US'
  AND date_in_utc >= DATE '2026-01-01'
GROUP BY visitor_id
HAVING MAX(COALESCE(request_7d_flag, 0)) = 1
ORDER BY signup_ts
LIMIT 5
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,visitor_id,signup_ts,first_fix_request_7d_flag
0,152514a4-ab55-4a44-bb81-a0c02d85f792,2011-10-30 07:00:00.000,1
1,13500a50-8116-4341-a38f-d515c24cc1cb,2011-10-30 07:00:00.000,1
2,e8e82847-1fe2-4b7e-b937-a3927e75321d,2011-10-30 07:00:00.000,1
3,b8a35d8e-acc1-468b-ab7c-4c1e36bb9aba,2011-10-30 07:00:00.000,1
4,00711f26-4cbd-4392-bf0a-ed8cb910c3db,2011-10-30 07:00:00.000,1


In [9]:
query("""--sql
SELECT
  COUNT(*) AS n_visitors,
  SUM(CASE WHEN signup_ts IS NOT NULL THEN 1 ELSE 0 END) AS n_with_signup,
  SUM(first_fix_request_7d_flag) AS n_with_first_fix_request
FROM (
  SELECT visitor_id, MAX(signup_ts) AS signup_ts, MAX(COALESCE(request_7d_flag, 0)) AS first_fix_request_7d_flag
  FROM curated.user_session_conversion_metrics
  WHERE region = 'US'
    AND date_in_utc >= DATE '2026-01-01'
  GROUP BY visitor_id
)
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,n_visitors,n_with_signup,n_with_first_fix_request
0,62064419,6331451,1420485


~1.4M visitors carry the flag at some point — a much smaller slice than the ~6.3M who ever signed up (A2), which is expected: requesting a Fix is a deeper funnel step than signing up, so this is a strict subset. There's no raw "request timestamp" column on this table (only the precomputed 7-day flag), so unlike the signup flag in Part A — which is recomputed from a raw `signup_ts` anchored to the exact signup-page-visit timestamp — this flag is taken at face value rather than re-derived. That's a known simplification, worth confirming with a table owner if this baseline needs to be pinned down further.

### B2 — the `classified` CTE (First Fix version)

Same join shape as A3 — `signup_page_visits` left-joined to the conversion source, `new_visitor` computed the same way — but the second flag is now `first_fix_request` instead of `signup`. Run on the **same 5 example visitors** as A3, for a direct before/after comparison:

In [10]:
query(f"""--sql
WITH signup_page_visits AS (
    SELECT visitor_id, DATE_TRUNC('month', datetime_in_utc) AS month, MIN(datetime_in_utc) AS signup_page_ts
    FROM curated.product_tracking_events
    WHERE date_in_utc >= DATE '2026-01-01'
      AND url LIKE 'https://www.stitchfix.com/signup%'
      AND visitor_id IN ('4a4eda80-25ef-4e13-bc8c-1f9a1ccc66e7', '15a3e35b-3505-4448-bd24-b11f0f715372', 'c7fff5a0-6feb-491e-824d-09bdd702c86e', '0f1f2a01-4bde-40bf-a211-e3b4a040d8ff', '9b4b1515-af03-4221-9410-b2f7dcf762c6')
    GROUP BY visitor_id, DATE_TRUNC('month', datetime_in_utc)
),
visitor_conversion AS (
    SELECT visitor_id, MAX(signup_ts) AS signup_ts, MAX(COALESCE(request_7d_flag, 0)) AS first_fix_request_7d_flag
    FROM curated.user_session_conversion_metrics
    WHERE region = 'US' AND date_in_utc >= DATE '2026-01-01'
      AND visitor_id IN ('4a4eda80-25ef-4e13-bc8c-1f9a1ccc66e7', '15a3e35b-3505-4448-bd24-b11f0f715372', 'c7fff5a0-6feb-491e-824d-09bdd702c86e', '0f1f2a01-4bde-40bf-a211-e3b4a040d8ff', '9b4b1515-af03-4221-9410-b2f7dcf762c6')
    GROUP BY visitor_id
)
SELECT
    v.visitor_id, v.signup_page_ts, c.signup_ts,
    CASE WHEN c.signup_ts IS NULL OR c.signup_ts >= v.signup_page_ts THEN 1 ELSE 0 END AS new_visitor,
    COALESCE(c.first_fix_request_7d_flag, 0) AS first_fix_request
FROM signup_page_visits v
LEFT JOIN visitor_conversion c ON c.visitor_id = v.visitor_id
ORDER BY v.signup_page_ts
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,visitor_id,signup_page_ts,signup_ts,new_visitor,first_fix_request
0,4a4eda80-25ef-4e13-bc8c-1f9a1ccc66e7,2026-01-01 00:00:03.030,2022-05-13 22:52:03.740,0,0
1,15a3e35b-3505-4448-bd24-b11f0f715372,2026-01-01 00:00:03.750,None,1,0
2,c7fff5a0-6feb-491e-824d-09bdd702c86e,2026-01-01 00:00:22.407,None,1,0
3,0f1f2a01-4bde-40bf-a211-e3b4a040d8ff,2026-01-01 00:00:25.349,2026-01-01 00:02:00.184,1,0
4,9b4b1515-af03-4221-9410-b2f7dcf762c6,2026-01-01 00:00:32.175,2026-01-01 00:00:46.555,1,0


Same `new_visitor` values as A3 (that part of the logic is untouched), but `first_fix_request = 0` for all five — none of these particular visitors requested a Fix within the window. That's expected, not a bug: two of the five never even signed up, and the two who did sign up did so within minutes, leaving little time to also complete Style Profile and request a Fix inside the same 7-day window. B1's aggregate count (~1.4M positives system-wide) confirms the flag does fire elsewhere; it's just genuinely rare at the individual-visitor level, consistent with First Fix's much lower conversion rate than signup.

### B3 — the full First Fix conversion query

Same structure as A4 — swap `visitor_signup`/`signup` for `visitor_conversion`/`first_fix_request`, keep the same `new_visitor` gate and `month_days` daily-rate denominator.

In [11]:
first_fix_query = """--sql
WITH signup_page_visits AS (
    SELECT
        visitor_id,
        DATE_TRUNC('month', datetime_in_utc) AS month,
        MIN(datetime_in_utc) AS signup_page_ts
    FROM curated.product_tracking_events
    WHERE date_in_utc >= DATE '2026-01-01'
      AND url LIKE 'https://www.stitchfix.com/signup%'
    GROUP BY visitor_id, DATE_TRUNC('month', datetime_in_utc)
),
visitor_conversion AS (
    SELECT
        visitor_id,
        MAX(signup_ts) AS signup_ts,
        MAX(COALESCE(request_7d_flag, 0)) AS first_fix_request_7d_flag
    FROM curated.user_session_conversion_metrics
    WHERE region = 'US'
      AND date_in_utc >= DATE '2026-01-01'
    GROUP BY visitor_id
),
classified AS (
    SELECT
        v.month,
        v.signup_page_ts,
        CASE WHEN c.signup_ts IS NULL OR c.signup_ts >= v.signup_page_ts
             THEN 1 ELSE 0 END AS new_visitor,
        COALESCE(c.first_fix_request_7d_flag, 0) AS first_fix_request
    FROM signup_page_visits v
    LEFT JOIN visitor_conversion c ON c.visitor_id = v.visitor_id
),
month_days AS (
    SELECT month, COUNT(DISTINCT CAST(signup_page_ts AS DATE)) AS days_observed
    FROM signup_page_visits
    GROUP BY month
)
SELECT
    c.month,
    md.days_observed,
    SUM(new_visitor) AS signup_page_visitors,
    SUM(CASE WHEN new_visitor = 1 THEN first_fix_request ELSE 0 END) AS first_fix_requests,
    CAST(SUM(CASE WHEN new_visitor = 1 THEN first_fix_request ELSE 0 END) AS DOUBLE)
        / NULLIF(SUM(new_visitor), 0) AS first_fix_conv_rate,
    ROUND(SUM(new_visitor) / md.days_observed, 1) AS signup_page_visitors_per_day
FROM classified c
JOIN month_days md ON c.month = md.month
GROUP BY c.month, md.days_observed
ORDER BY c.month DESC
"""

query(first_fix_query)

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,month,days_observed,signup_page_visitors,first_fix_requests,first_fix_conv_rate,signup_page_visitors_per_day
0,2026-07-01 00:00:00.000,28,256861,25338,0.098645,9173
1,2026-06-01 00:00:00.000,30,225286,22614,0.100379,7509
2,2026-05-01 00:00:00.000,31,269155,30122,0.111913,8682
3,2026-04-01 00:00:00.000,30,288796,33134,0.114732,9626
4,2026-03-01 00:00:00.000,31,368711,42713,0.115844,11893
5,2026-02-01 00:00:00.000,28,335704,36061,0.107419,11989
6,2026-01-01 00:00:00.000,31,358841,37967,0.105805,11575


First Fix conversion sits around **10–12%** of new signup-page visitors — a fifth of the signup rate in Part A, which tracks: requesting a Fix requires signing up, completing Style Profile, *and* requesting, all within the same 7-day window, versus just signing up.